In [93]:
# Imports
import anthropic
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import sys
sys.path.append('..')
import prompts
import utils

In [ ]:
load_dotenv(override=True)

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
anthropic_model = os.getenv("ANTHROPIC_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("BASIC_OUTPUT_FOLDER_PATH")

In [ ]:
def anthropic_nutritionist(image):
    try:
        response = client.messages.create(
            model=anthropic_model,  # Claude 3 vision model
            temperature=0.1,
            max_tokens=1024,
            system=prompts.SYSTEM_PROMPT_NUTRITIONIST,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": f"{image}"
                            }
                        },
                        {
                            "type": "text",
                            "text": prompts.UNIFIED_NUTRITION_PROMPT
                        }
                    ]
                }
            ]
        )

        data_text = "".join(
            block.text for block in response.content if block.type == "text"
        )

        data = utils.parse_json(data_text)

        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }

    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
def analyze_image(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)

    with open(image_path, "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")

    print(f"\n🔄 Processing {index}: {file_name}")

    print("  Anthropic Nutritionist...")
    response = anthropic_nutritionist(image_base64)

    if not response['success']:
        return {'success': False, 'error': f"Anthropic failed: {response['error']}", 'index': index}
    
    print(f"    → {response['success']}")
    if not response.get("description"):
        return {'success': False, 'error': "Anthropic returned empty description", 'index': index}

    total_time = time.time() - start_time
    time.sleep(2)
    print(f"  ✅ Complete! {response['calories']} in {total_time:.1f}s")

    return {
        'success': True,
        'id': index,
        'file_name': file_name,
        'response': response,
        'processing_time': total_time
    }

In [ ]:
# Process dataset
def process_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""
    
    total_images = utils.count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [90]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['response']
            final_results.append({
                'id': item['index'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [87]:
file_name = "nutria_anthropic"

In [ ]:
results = process_dataset(file_name, images_folder_path, start=1, end=utils.count_images(images_folder_path))
# results = process_chained_dataset(file_name, images_folder_path, start=1, end=2)

🚀 Processing images 1 to 50 (50 total)

🔄 Processing 1: 1.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 215.0 kcal in 3.3s

🔄 Processing 2: 2.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 312.0 kcal in 3.4s

🔄 Processing 3: 3.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 520.0 kcal in 4.6s

🔄 Processing 4: 4.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 520.0 kcal in 3.7s

🔄 Processing 5: 5.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 420.0 kcal in 4.3s

🔄 Processing 6: 6.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 420.0 kcal in 4.4s

🔄 Processing 7: 7.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 245.0 kcal in 3.4s

🔄 Processing 8: 8.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 2.0 kcal in 4.0s

🔄 Processing 9: 9.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 42.0 kcal in 10.6s

🔄 Processing 10: 10.jpg
  Anthropic Nutritionist...
    → True
  ✅ Complete! 0.0 kcal in 8.8s

🔄 Processing

In [91]:
excel_path = export_to_excel(file_name)
print(f"\n🎯 Done! Check: {excel_path}")

✅ Excel exported: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_anthropic.xlsx
📊 48 successful analyses

🎯 Done! Check: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_anthropic.xlsx
